2. Spark background

○ What is Spark?

Apache spark is an open source distributed framework used for processing very large datasets quickly. It allows for data to be split across across multiple machines and process in parallel instead of relying on just one computer.Spark helps us work with massive amounts of data uch faster than traditional single machine tools.

○ Why is Spark a popular framework?

Spark is a popular framework because it is fast,flexible and easy to use compared to many older big data tools.

Main reasons for its popularity are -

1. Speed: Spark performs many operations in memory which makes it much faster than older disk based systems like traditional Hadoop MapReduce.

2. Distributed processing: It can handle huge datasets by dividing work across many machines.

3. Multiple libraries in one ecosystem: Spark supports SQL, machine learning, streaming and graph analytics in the same framework.

4. Ease of Use : It provides high level API's in python, Scala, Java and R so developers can write less code.

5. Scalability - It works on small datasets on one machine and also scales to clusters with many nodes

6. Fault Tolerance - If one machine fails spark can recover and still continue processing.

○ What is Spark SQL and why does it exist?

Spark SQL is a spark module used for working with structured and semi structured data using SQL queries or DataFrame operations.

It exists because many users are already familiar with SQL and a large amount of real world data is stored in tables, CSV files,JSON files and databases. Spark SQL makes it easier to analyze this kind of data without writing low level distributed code.

Why spark SQL exists:

1. To let users query big data using familiar SQL syntax.

2. To simplify strucuted data analysis.

3. To integrate relational processing with Spark's distributed engine.

4. To optimize queries automatically for better performance

○ What is a Spark DataFrame, and why is it useful?

A spark DataFrame is a distributed collection of data organized into named columns,similar to a table in a databaseor a pandas DataFrame.

It is useful becuase it gives a simple and structured way to work with large datasets while still using Spark's distributed processing power.

Why Spark DataFrames are useful:

1. Structured format: Data is organized into rows and columns which makes it easy to understand.

2. Easy transformations: You can filter,group,join and aggregate data easily.

3. Optimized execution: Spark automatically optimizes DataFrame operations, making them efficient.

4. Works with many data sources: DataFrames can read from CSV,JSON,Hive etc.

5. Integration with Spark SQL: You can use both SQL queries and DataFrame methods on the same data.

6. Sacalable : They work on very large datasets aqcross clusters.

3. Setup / Data Collection

○ Colab should have PySpark already installed (else: pip install pyspark)

○ In your Colab, provide a link to the Kaggle dataset so that we can review and fetch
in case needed to run your code

#Dataset link:
https://www.kaggle.com/datasets/dubradave/hospital-readmissions

○ Add the actual dataset to your Colab notebook (either fetch with wget or manually
add to your Colab files)

In [ ]:
import os
print(os.listdir('/content'))

['.config', 'hospital_readmissions.csv', 'sample_data']


Instantiating a spark session

In [ ]:
!pip install -q pyspark

Creating a spark session

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("HospitalReadmissionsSpark") \
    .getOrCreate()
print("Spark session created successfully")
print("Spark version:", spark.version)

Spark session created successfully
Spark version: 3.5.1


Importing the csv as a spark dataframe

In [ ]:
df_new = spark.read.csv("/content/hospital_readmissions.csv",
                    header=True,
                    inferSchema=True
)


 4. Data Cleaning

 Using Spark SQL, check for any potential issues in the data from Step 3 and address as needed.

■ Are there duplicates? How many?

■ Any missing or null values? If so, decide how you want to handle them and handle them.

■ Any type inconsistencies?

■ Any problematic misspellings /mistypings?

■ Any outliers? Check each column (no more than 10 columns)

In [ ]:
#First inspecting the dataset
df_new.printSchema()
print(df_new.columns)
df_new.show(5,truncate=False)

root
 |-- age: string (nullable = true)
 |-- time_in_hospital: integer (nullable = true)
 |-- n_lab_procedures: integer (nullable = true)
 |-- n_procedures: integer (nullable = true)
 |-- n_medications: integer (nullable = true)
 |-- n_outpatient: integer (nullable = true)
 |-- n_inpatient: integer (nullable = true)
 |-- n_emergency: integer (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- glucose_test: string (nullable = true)
 |-- A1Ctest: string (nullable = true)
 |-- change: string (nullable = true)
 |-- diabetes_med: string (nullable = true)
 |-- readmitted: string (nullable = true)

['age', 'time_in_hospital', 'n_lab_procedures', 'n_procedures', 'n_medications', 'n_outpatient', 'n_inpatient', 'n_emergency', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'glucose_test', 'A1Ctest', 'change', 'diabetes_med', 'readmitted']
+-------+-------------

In [ ]:
#Creating a temporary SQl view
df_new.createOrReplaceTempView("hospital_data")

In [ ]:
#Checking for duplicate values
#Counting total rows
spark.sql("""
SELECT COUNT(*) AS total_rows
FROM hospital_data
""").show()

#Count distinct rows
spark.sql("""
SELECT COUNT(*) AS distinct_rows
FROM (
  SELECT DISTINCT *
  from hospital_data
)
""").show()
#Simpler duplicate count
total_count = df_new.count()
distinct_count = df_new.distinct().count()
print("Duplicate rows:", total_count - distinct_count)


+----------+
|total_rows|
+----------+
|     25000|
+----------+

+-------------+
|distinct_rows|
+-------------+
|        25000|
+-------------+

Duplicate rows: 0


I checked for duplicate rows in the dataset using spark and found that the number of duplicate rows was 0. Therefore no duplicate removal is neccesary.


In [ ]:
df_new.columns

['age',
 'time_in_hospital',
 'n_lab_procedures',
 'n_procedures',
 'n_medications',
 'n_outpatient',
 'n_inpatient',
 'n_emergency',
 'medical_specialty',
 'diag_1',
 'diag_2',
 'diag_3',
 'glucose_test',
 'A1Ctest',
 'change',
 'diabetes_med',
 'readmitted']

In [ ]:
#Checking null or missing values

spark.sql("""
SELECT
SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS age_nulls,
SUM(CASE WHEN time_in_hospital IS NULL THEN 1 ELSE 0 END) AS time_in_hospital_nulls,
SUM(CASE WHEN n_lab_procedures IS NULL THEN 1 ELSE 0 END) AS n_lab_procedures_nulls,
SUM(CASE WHEN n_procedures IS NULL THEN 1 ELSE 0 END) AS n_procedures_nulls,
SUM(CASE WHEN n_medications IS NULL THEN 1 ELSE 0 END) AS n_medications_nulls,
SUM(CASE WHEN n_outpatient IS NULL THEN 1 ELSE 0 END) AS n_out_patient_nulls,
SUM(CASE WHEN n_inpatient IS NULL THEN 1 ELSE 0 END) AS n_inpatient_nulls,
SUM(CASE WHEN n_emergency IS NULL THEN 1 ELSE 0 END) AS n_emergency_nulls,
SUM(CASE WHEN medical_specialty IS NULL OR TRIM(medical_specialty) = '' THEN 1 ELSE 0 END) AS medical_specialty_nulls,
SUM(CASE WHEN diag_1 IS NULL OR TRIM(diag_1) = '' THEN 1 ELSE 0 END) AS diag_1_nulls,
SUM(CASE WHEN diag_2 IS NULL OR TRIM(diag_2) = '' THEN 1 ELSE 0 END) AS diag_2_nulls,
SUM(CASE WHEN diag_3 IS NULL OR TRIM(diag_3) = '' THEN 1 ELSE 0 END) AS diag_3_nulls,
SUM(CASE WHEN glucose_test IS NULL OR TRIM(glucose_test)='' THEN 1 ELSE 0 END) AS glucose_test_nulls,
SUM(CASE WHEN A1Ctest IS NULL OR TRIM(A1Ctest) = '' THEN 1 ELSE 0 END) AS A1Ctest_nulls,
SUM(CASE WHEN change IS NULL OR TRIM(change) = '' THEN 1 ELSE 0 END) AS change_nulls,
SUM(CASE WHEN diabetes_med IS NULL OR TRIM(diabetes_med) = '' THEN 1 ELSE 0 END) AS diabetes_med_nulls,
SUM(CASE WHEN readmitted IS NULL OR TRIM(readmitted) = '' THEN 1 ELSE 0 END) AS readmitted_nulls
FROM hospital_data
""").show(truncate=False)

+---------+----------------------+----------------------+------------------+-------------------+-------------------+-----------------+-----------------+-----------------------+------------+------------+------------+------------------+-------------+------------+------------------+----------------+
|age_nulls|time_in_hospital_nulls|n_lab_procedures_nulls|n_procedures_nulls|n_medications_nulls|n_out_patient_nulls|n_inpatient_nulls|n_emergency_nulls|medical_specialty_nulls|diag_1_nulls|diag_2_nulls|diag_3_nulls|glucose_test_nulls|A1Ctest_nulls|change_nulls|diabetes_med_nulls|readmitted_nulls|
+---------+----------------------+----------------------+------------------+-------------------+-------------------+-----------------+-----------------+-----------------------+------------+------------+------------+------------------+-------------+------------+------------------+----------------+
|0        |0                     |0                     |0                 |0                  |0         

I checked for null/missing values in the above dataset and found out that no column has any missing values.

In [ ]:
#Checking type incosistencies
#Checking data types of all columns
df_new.printSchema()

root
 |-- age: string (nullable = true)
 |-- time_in_hospital: integer (nullable = true)
 |-- n_lab_procedures: integer (nullable = true)
 |-- n_procedures: integer (nullable = true)
 |-- n_medications: integer (nullable = true)
 |-- n_outpatient: integer (nullable = true)
 |-- n_inpatient: integer (nullable = true)
 |-- n_emergency: integer (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- glucose_test: string (nullable = true)
 |-- A1Ctest: string (nullable = true)
 |-- change: string (nullable = true)
 |-- diabetes_med: string (nullable = true)
 |-- readmitted: string (nullable = true)



In [ ]:
#Creating numeric midpoint age
from pyspark.sql.functions import regexp_extract, col
df_new = df_new.withColumn(
    "age_mid",
    (
        regexp_extract(col("age"), r"\[(\d+)-", 1).cast("int") +
        regexp_extract(col("age"), r"-(\d+)\)", 1).cast("int")
    ) /2.0
)

In [ ]:
#Creating a sql view from the new dataframe
df_new.createOrReplaceTempView("hospital_data")

I checked the schema using printSchema() and found one type inconsistency in the dataset. The age column was stored as a string instead of integer while the other numeric columns were already in integer format.

In [ ]:
# Any problamatic misspellings/misstypings
#checking for each column

In [ ]:
#medical specialty
spark.sql("""
SELECT medical_specialty, COUNT(*) AS count
FROM hospital_data
GROUP BY medical_specialty
ORDER BY count DESC
""").show(50, truncate=False)


+----------------------+-----+
|medical_specialty     |count|
+----------------------+-----+
|Missing               |12382|
|InternalMedicine      |3565 |
|Other                 |2664 |
|Emergency/Trauma      |1885 |
|Family/GeneralPractice|1882 |
|Cardiology            |1409 |
|Surgery               |1213 |
+----------------------+-----+



In [ ]:
#Diag_1
spark.sql("""
SELECT diag_1, COUNT(*) AS count
FROM hospital_data
GROUP BY diag_1
ORDER BY count DESC
""").show(50, truncate = False)

+---------------+-----+
|diag_1         |count|
+---------------+-----+
|Circulatory    |7824 |
|Other          |6498 |
|Respiratory    |3680 |
|Digestive      |2329 |
|Diabetes       |1747 |
|Injury         |1666 |
|Musculoskeletal|1252 |
|Missing        |4    |
+---------------+-----+



In [ ]:
#diag_2
spark.sql("""
SELECT diag_2, COUNT(*) AS count
FROM hospital_data
GROUP BY diag_2
ORDER BY count DESC
""").show(50, truncate=False)

+---------------+-----+
|diag_2         |count|
+---------------+-----+
|Other          |9056 |
|Circulatory    |8134 |
|Diabetes       |2906 |
|Respiratory    |2872 |
|Digestive      |973  |
|Injury         |591  |
|Musculoskeletal|426  |
|Missing        |42   |
+---------------+-----+



In [ ]:
#diag_3
spark.sql("""
SELECT diag_3, COUNT(*) AS count
FROM hospital_data
GROUP BY diag_3
ORDER BY count DESC
""").show(50, truncate=False)

+---------------+-----+
|diag_3         |count|
+---------------+-----+
|Other          |9107 |
|Circulatory    |7686 |
|Diabetes       |4261 |
|Respiratory    |1915 |
|Digestive      |916  |
|Injury         |464  |
|Musculoskeletal|455  |
|Missing        |196  |
+---------------+-----+



In [ ]:
#Glucose_test
spark.sql("""
SELECT glucose_test, COUNT(*) AS count
FROM hospital_data
GROUP BY glucose_test
ORDER BY count DESC
""").show(truncate=False)

+------------+-----+
|glucose_test|count|
+------------+-----+
|no          |23625|
|normal      |689  |
|high        |686  |
+------------+-----+



In [ ]:
#A1Ctest
spark.sql("""
SELECT A1Ctest, COUNT(*) AS count
FROM hospital_data
GROUP BY A1Ctest
ORDER BY count DESC
""").show(truncate= False)

+-------+-----+
|A1Ctest|count|
+-------+-----+
|no     |20938|
|high   |2827 |
|normal |1235 |
+-------+-----+



In [ ]:
#change
spark.sql("""
SELECT change, COUNT(*) AS count
from hospital_data
GROUP BY change
ORDER BY count DESC
""").show(truncate=False)

+------+-----+
|change|count|
+------+-----+
|no    |13497|
|yes   |11503|
+------+-----+



In [ ]:
#Diabetes_med
spark.sql("""
SELECT diabetes_med, COUNT(*) AS count
from hospital_data
GROUP BY diabetes_med
ORDER BY count DESC
""").show(truncate=False)

+------------+-----+
|diabetes_med|count|
+------------+-----+
|yes         |19228|
|no          |5772 |
+------------+-----+



In [ ]:
#Readmitted
spark.sql("""
SELECT readmitted, COUNT(*) AS count
FROM hospital_data
GROUP BY readmitted
ORDER BY count DESC
""").show(truncate=False)

+----------+-----+
|readmitted|count|
+----------+-----+
|no        |13246|
|yes       |11754|
+----------+-----+



After checking all the columns for inconsistencies we found out that 4 columns contain missing a placeholder value for unavailable information. Since these are literal strings, null could not identify these values. Hence we need to convert Missing to null and then remove those rows.

In [ ]:
from pyspark.sql.functions import when, col
#Converting missing to null in selected columns
df_new = df_new.withColumn(
    "medical_specialty",
    when(col("medical_specialty")== "Missing", None).otherwise(col("medical_specialty"))
).withColumn(
    "diag_1",
    when(col("diag_1")== "Missing", None).otherwise(col("diag_1"))
).withColumn(
    "diag_2",
    when(col("diag_2") == "Missing", None).otherwise(col("diag_2"))
).withColumn(
    "diag_3",
    when(col("diag_3")=="Missing",None).otherwise(col("diag_3"))
)

#Dropping rows where any of these columns become null
df_new = df_new.dropna(subset=["medical_specialty", "diag_1", "diag_2", "diag_3"])

#Recreating SQL View
df_new.createOrReplaceTempView("hospital_data")

In [ ]:
# Outliers
#Selecting columns for outlier analysis
numeric_cols = [
    "age_mid",
    "time_in_hospital",
    "n_lab_procedures",
    "n_procedures",
    "n_medications",
    "n_outpatient",
    "n_inpatient",
    "n_emergency"
]

print("Columns chosen for outlier detection:", numeric_cols)

Columns chosen for outlier detection: ['age_mid', 'time_in_hospital', 'n_lab_procedures', 'n_procedures', 'n_medications', 'n_outpatient', 'n_inpatient', 'n_emergency']


In [ ]:
#Viewing basic summary statistics before checking for outliers
spark.sql("""
SELECT
  MIN(age_mid) as min_age_mid, MAX(age_mid) AS max_age_mid, AVG(age_mid) AS avg_age_mid,
  MIN(time_in_hospital) AS min_time_in_hospital, MAX(time_in_hospital) AS max_time_in_hospital, AVG(time_in_hospital) AS avg_time_in_hospital,
  MIN(n_lab_procedures) AS min_n_lab_procedures, MAX(n_procedures) AS max_n_procedures, AVG(n_procedures) AS avg_n_lab_procedures,
  MIN(n_procedures) AS min_n_procedures, MAX(n_procedures) AS max_n_procedures, AVG(n_procedures) AS avg_n_procedures,
  MIN(n_medications) AS min_n_medications, MAX(n_medications) AS max_n_medications, AVG(n_medications) AS avg_n_medications,
  MIN(n_outpatient) AS min_n_outpatient, MAX(n_outpatient) AS max_n_outpatient, AVG(n_outpatient) AS avg_n_outpatient,
  MIN(n_inpatient) AS min_n_patient, MAX(n_inpatient) AS max_n_inpatient, AVG(n_inpatient) AS avg_n_inpatient,
  MIN(n_emergency) AS min_n_emergency, MAX(n_emergency) as max_n_emergency, AVG(n_emergency) AS avg_n_emergency
FROM hospital_data
""").show(truncate=False)

+-----------+-----------+-----------------+--------------------+--------------------+--------------------+--------------------+----------------+--------------------+----------------+----------------+------------------+-----------------+-----------------+-----------------+----------------+----------------+-----------------+-------------+---------------+------------------+---------------+---------------+-------------------+
|min_age_mid|max_age_mid|avg_age_mid      |min_time_in_hospital|max_time_in_hospital|avg_time_in_hospital|min_n_lab_procedures|max_n_procedures|avg_n_lab_procedures|min_n_procedures|max_n_procedures|avg_n_procedures  |min_n_medications|max_n_medications|avg_n_medications|min_n_outpatient|max_n_outpatient|avg_n_outpatient |min_n_patient|max_n_inpatient|avg_n_inpatient   |min_n_emergency|max_n_emergency|avg_n_emergency    |
+-----------+-----------+-----------------+--------------------+--------------------+--------------------+--------------------+----------------+----

Computing Q1,Q3, IQR, lower bound and upper bound for each column.
I will use the inter quartile range(IQR) method to detect outliers.

IQR = Q3 - Q1

Lower Bound = Q1 - 1.5 * IQR

Upper Bound = Q3 + 1.5 * IQR


In [ ]:
hospital_df = spark.table("hospital_data")
outlier_bounds = {}
for c in numeric_cols:
  q1, q3 = hospital_df.approxQuantile(c, [0.25, 0.75], 0.01)
  iqr = q3 - q1
  lower = q1 - 1.5 * iqr
  upper = q3 + 1.5* iqr

  outlier_bounds[c] = {
      "Q1": q1,
      "Q3": q3,
      "IQR": iqr,
      "Lower_Bound": lower,
      "Upper_Bound": upper
  }

  print(f"{c}")
  print(f" Q1 = {q1}")
  print(f" Q3 = {q3}")
  print(f" IQR = {iqr}")
  print(f" Lower Bound = {lower}")
  print(f" Upper Bound = {upper}")

age_mid
 Q1 = 55.0
 Q3 = 75.0
 IQR = 20.0
 Lower Bound = 25.0
 Upper Bound = 105.0
time_in_hospital
 Q1 = 2.0
 Q3 = 6.0
 IQR = 4.0
 Lower Bound = -4.0
 Upper Bound = 12.0
n_lab_procedures
 Q1 = 33.0
 Q3 = 56.0
 IQR = 23.0
 Lower Bound = -1.5
 Upper Bound = 90.5
n_procedures
 Q1 = 0.0
 Q3 = 2.0
 IQR = 2.0
 Lower Bound = -3.0
 Upper Bound = 5.0
n_medications
 Q1 = 11.0
 Q3 = 20.0
 IQR = 9.0
 Lower Bound = -2.5
 Upper Bound = 33.5
n_outpatient
 Q1 = 0.0
 Q3 = 0.0
 IQR = 0.0
 Lower Bound = 0.0
 Upper Bound = 0.0
n_inpatient
 Q1 = 0.0
 Q3 = 1.0
 IQR = 1.0
 Lower Bound = -1.5
 Upper Bound = 2.5
n_emergency
 Q1 = 0.0
 Q3 = 0.0
 IQR = 0.0
 Lower Bound = 0.0
 Upper Bound = 0.0


In [ ]:
#Counting how many outliers each column has
for c, vals in outlier_bounds.items():
  lower = vals["Lower_Bound"]
  upper = vals["Upper_Bound"]
  print(f"Checking outliers in {c}")

  spark.sql(f"""
  SELECT COUNT(*) AS outlier_count
  FROM hospital_data
  WHERE {c} < {lower} OR {c} > {upper}
  """).show()

Checking outliers in age_mid
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+

Checking outliers in time_in_hospital
+-------------+
|outlier_count|
+-------------+
|          307|
+-------------+

Checking outliers in n_lab_procedures
+-------------+
|outlier_count|
+-------------+
|           63|
+-------------+

Checking outliers in n_procedures
+-------------+
|outlier_count|
+-------------+
|          619|
+-------------+

Checking outliers in n_medications
+-------------+
|outlier_count|
+-------------+
|          442|
+-------------+

Checking outliers in n_outpatient
+-------------+
|outlier_count|
+-------------+
|         1562|
+-------------+

Checking outliers in n_inpatient
+-------------+
|outlier_count|
+-------------+
|          754|
+-------------+

Checking outliers in n_emergency
+-------------+
|outlier_count|
+-------------+
|         1326|
+-------------+



In [ ]:
#Showing the actual outlier values for each column
for c , vals in outlier_bounds.items():
  lower = vals["Lower_Bound"]
  upper = vals["Upper_Bound"]

  print(f"Outlier Values in {c}:")
  spark.sql(f"""
  SELECT {c}
  FROM hospital_data
  WHERE {c} < {lower} OR {c} > {upper}
  ORDER BY {c}
  """).show(20, truncate=False)

Outlier Values in age_mid:
+-------+
|age_mid|
+-------+
+-------+

Outlier Values in time_in_hospital:
+----------------+
|time_in_hospital|
+----------------+
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
|13              |
+----------------+
only showing top 20 rows

Outlier Values in n_lab_procedures:
+----------------+
|n_lab_procedures|
+----------------+
|91              |
|91              |
|91              |
|91              |
|91              |
|91              |
|92              |
|92              |
|92              |
|92              |
|92              |
|92              |
|92              |
|92              |
|93              |
|93              |
|93              

In [ ]:
#Checking for impossible values such as
#age, hospital visits should not be negative, etc.

spark.sql("""
SELECT *
FROM hospital_data
WHERE age_mid < 0
OR time_in_hospital < 0
OR n_lab_procedures < 0
OR n_medications < 0
OR n_procedures < 0
OR n_medications < 0
OR n_outpatient < 0
OR n_inpatient < 0
OR n_emergency < 0
""").show(truncate=False)

+---+----------------+----------------+------------+-------------+------------+-----------+-----------+-----------------+------+------+------+------------+-------+------+------------+----------+-------+
|age|time_in_hospital|n_lab_procedures|n_procedures|n_medications|n_outpatient|n_inpatient|n_emergency|medical_specialty|diag_1|diag_2|diag_3|glucose_test|A1Ctest|change|diabetes_med|readmitted|age_mid|
+---+----------------+----------------+------------+-------------+------------+-----------+-----------+-----------------+------+------+------+------------+-------+------+------------+----------+-------+
+---+----------------+----------------+------------+-------------+------------+-----------+-----------+-----------------+------+------+------+------------+-------+------+------------+----------+-------+



In [ ]:
#Creating a capped dataset that will keep all the rows but cap
# the extreme values

from pyspark.sql.functions import when, col
df_capped = df_new
for c, vals in outlier_bounds.items():
  lower = vals["Lower_Bound"]
  upper = vals["Upper_Bound"]

  df_capped = df_capped.withColumn(
      c,
      when(col(c) < lower, lower)
      .when(col(c) > upper, upper)
      .otherwise(col(c))
  )
  df_capped.createOrReplaceTempView("hospital_data_capped")

In [ ]:
#Rechecking outlier counts after capping
for c, vals in outlier_bounds.items():
  lower=vals["Lower_Bound"]
  upper = vals["Upper_Bound"]
  print(f"Checking capped outliers in {c}")

  spark.sql(f"""
  SELECT COUNT(*) AS outlier_count
  FROM hospital_data_capped
  WHERE {c} < {lower} OR {c} > {upper}
            """).show()

Checking capped outliers in age_mid
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+

Checking capped outliers in time_in_hospital
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+

Checking capped outliers in n_lab_procedures
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+

Checking capped outliers in n_procedures
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+

Checking capped outliers in n_medications
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+

Checking capped outliers in n_outpatient
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+

Checking capped outliers in n_inpatient
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+

Checking capped outliers in n_emergency
+-------------+
|outlier_count|
+-------------+
|            0|
+-------------+



5. Data Exploration

○ Explore characteristics of the data from Step 3 using Spark SQL.

In [ ]:
#Setting the updated dataframe and creating SQL view

#Choosing the updated dataframe and create SQL view.
df_explore = df_capped

#Register temp view for Spark SQL

df_explore.createOrReplaceTempView("hospital_data_explore")


In [ ]:
#Printing the schema
df_explore.printSchema()
print("Column names:", df_explore.columns)

root
 |-- age: string (nullable = true)
 |-- time_in_hospital: double (nullable = true)
 |-- n_lab_procedures: double (nullable = true)
 |-- n_procedures: double (nullable = true)
 |-- n_medications: double (nullable = true)
 |-- n_outpatient: double (nullable = true)
 |-- n_inpatient: double (nullable = true)
 |-- n_emergency: double (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- glucose_test: string (nullable = true)
 |-- A1Ctest: string (nullable = true)
 |-- change: string (nullable = true)
 |-- diabetes_med: string (nullable = true)
 |-- readmitted: string (nullable = true)
 |-- age_mid: double (nullable = true)

Column names: ['age', 'time_in_hospital', 'n_lab_procedures', 'n_procedures', 'n_medications', 'n_outpatient', 'n_inpatient', 'n_emergency', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'glucose_test', 'A1Ctest', 'change', 'diabe

I Used the updated cleaned dataset for data exploration. The schema shows the names and data types of each colum. Most hospital utiization columns are numeric, while diagnosis and treatment related columns are categorical strings.

In [ ]:
#Showing shape of the data
num_rows = df_explore.count()
num_cols = len(df_explore.columns)
print("Number of rows:", num_rows)
print("Number of columns:", num_cols)
print("Shape: ", (num_rows, num_cols))

Number of rows: 12465
Number of columns: 18
Shape:  (12465, 18)


The shape of the updated dataset is shown by counting the number of rows and columns. This helps confirm the final size of the cleaned dataset used for analysis.

In [ ]:
spark.sql("""
SELECT
    COUNT(age_mid) AS count_age,
    ROUND(AVG(age_mid), 2) AS mean_age,
    ROUND(STDDEV(age_mid),2) AS stddev_age,
    MIN(age_mid) AS min_age,
    MAX(age_mid) AS max_age,
    COUNT(time_in_hospital) AS count_time_in_hospital,
    ROUND(AVG(time_in_hospital), 2) AS mean_time_in_hospital,
    ROUND(STDDEV(time_in_hospital), 2) AS stddev_time_in_hospital,
    MIN(time_in_hospital) AS min_time_in_hospital,
    MAX(time_in_hospital) AS max_time_in_hospital
FROM hospital_data_explore
""").show(truncate=False)

+---------+--------+----------+-------+-------+----------------------+---------------------+-----------------------+--------------------+--------------------+
|count_age|mean_age|stddev_age|min_age|max_age|count_time_in_hospital|mean_time_in_hospital|stddev_time_in_hospital|min_time_in_hospital|max_time_in_hospital|
+---------+--------+----------+-------+-------+----------------------+---------------------+-----------------------+--------------------+--------------------+
|12465    |68.17   |13.18     |45.0   |95.0   |12465                 |4.47                 |2.94                   |1.0                 |12.0                |
+---------+--------+----------+-------+-------+----------------------+---------------------+-----------------------+--------------------+--------------------+



I generated summary statistics such as count, mean,standard deviation,minimum and maximum to understand the central tendency and spread of the numeric variables.

In [ ]:
#Distribution of age_mid
spark.sql("""
SELECT age_mid, COUNT(*) as count
FROM hospital_data_explore
GROUP BY age_mid
ORDER BY age_mid
""").show(truncate=False)

+-------+-----+
|age_mid|count|
+-------+-----+
|45.0   |1308 |
|55.0   |2291 |
|65.0   |2904 |
|75.0   |3448 |
|85.0   |2137 |
|95.0   |377  |
+-------+-----+



Meaning

1. Represents approximate patient age using the midpoint of each age interval.

2. Units: years

age_mid represnts approximate patient age in years. Its distribution shows which age groups are most common in the dataset. This is useful because age is an important factor n hospital Utilization and readmission risk.

In [ ]:
#Distribution of time_in_hospital
spark.sql("""
SELECT time_in_hospital, COUNT(*) AS count
FROM hospital_data_explore
GROUP BY time_in_hospital
ORDER BY time_in_hospital
""").show(truncate=False)

+----------------+-----+
|time_in_hospital|count|
+----------------+-----+
|1.0             |1790 |
|2.0             |1917 |
|3.0             |2069 |
|4.0             |1706 |
|5.0             |1277 |
|6.0             |952  |
|7.0             |775  |
|8.0             |559  |
|9.0             |386  |
|10.0            |296  |
|11.0            |247  |
|12.0            |491  |
+----------------+-----+



Meaning:
1. Number of days spent in hospital
2. Units: days

time_in_hospital represnts the number of hospital days. Its distribution helps show whether most patients have short stays or whether there are many long stay cases.

In [ ]:
spark.sql("""
SELECT n_lab_procedures, COUNT(*) AS count
FROM hospital_data_explore
GROUP BY n_lab_procedures
ORDER BY n_lab_procedures
""").show(50, truncate= False)

+----------------+-----+
|n_lab_procedures|count|
+----------------+-----+
|1.0             |370  |
|2.0             |113  |
|3.0             |89   |
|4.0             |36   |
|5.0             |36   |
|6.0             |31   |
|7.0             |35   |
|8.0             |50   |
|9.0             |125  |
|10.0            |99   |
|11.0            |71   |
|12.0            |66   |
|13.0            |38   |
|14.0            |39   |
|15.0            |50   |
|16.0            |77   |
|17.0            |82   |
|18.0            |92   |
|19.0            |96   |
|20.0            |104  |
|21.0            |67   |
|22.0            |95   |
|23.0            |105  |
|24.0            |90   |
|25.0            |116  |
|26.0            |116  |
|27.0            |109  |
|28.0            |121  |
|29.0            |171  |
|30.0            |125  |
|31.0            |165  |
|32.0            |185  |
|33.0            |159  |
|34.0            |246  |
|35.0            |256  |
|36.0            |236  |
|37.0            |255  |


Meaning
1. Number of lab procedures performed.
2. Units: count of procedures

n_lab_procedures represnts how many lab tests or lab procedures a patient received. The distribution helps identify the typical care intensity and whether a small number of patients required especially high testing.

In [ ]:
#Distribution of n_medications
spark.sql("""
SELECT n_medications, COUNT(*) AS count
FROM hospital_data_explore
GROUP BY n_medications
ORDER BY n_medications
""").show(50, truncate=False)

+-------------+-----+
|n_medications|count|
+-------------+-----+
|1.0          |37   |
|2.0          |42   |
|3.0          |115  |
|4.0          |148  |
|5.0          |222  |
|6.0          |323  |
|7.0          |442  |
|8.0          |568  |
|9.0          |618  |
|10.0         |652  |
|11.0         |719  |
|12.0         |766  |
|13.0         |699  |
|14.0         |705  |
|15.0         |722  |
|16.0         |647  |
|17.0         |582  |
|18.0         |566  |
|19.0         |467  |
|20.0         |463  |
|21.0         |374  |
|22.0         |355  |
|23.0         |313  |
|24.0         |257  |
|25.0         |213  |
|26.0         |182  |
|27.0         |195  |
|28.0         |147  |
|29.0         |132  |
|30.0         |112  |
|31.0         |101  |
|32.0         |76   |
|33.0         |63   |
|33.5         |442  |
+-------------+-----+



Meaning:
1. Number of medications used
2. Units: counts of medications

n_medications represents the number of medications prescribed or administered. The distribution helps show treatment complexity and whether some patients require unusually large medication counts.

In [ ]:
#Distrbtuion of n_outpatient
spark.sql("""
SELECT n_outpatient, COUNT(*) AS count
FROM hospital_data_explore
GROUP BY n_outpatient
ORDER BY n_outpatient
""").show(truncate=False)

+------------+-----+
|n_outpatient|count|
+------------+-----+
|0.0         |12465|
+------------+-----+



Meaning
1. Number of outpatient visits
2. Units: visit count

n_outpatient represents the number of outpatient visits. The distribution shows whether most patients had no outpatient history or whether some patients had repeated visits before hospitalization

In [ ]:
#Distribution of target column readmitted
spark.sql("""
SELECT readmitted, COUNT(*) AS count
FROM hospital_data_explore
GROUP BY readmitted
ORDER BY count DESC
""").show(truncate =False)

+----------+-----+
|readmitted|count|
+----------+-----+
|no        |6810 |
|yes       |5655 |
+----------+-----+



Meaning
1. Whether the patient was readmitted
2. Units: category/label

readmitted is the target label for the machine learning task. Its distribution is useful because it shows class balance which affects how we interpret model performance later

In [ ]:
#Exploring the relationship between time_in_hospital and readmitted

spark.sql("""
SELECT
    readmitted,
    ROUND (AVG(time_in_hospital), 2) AS avg_time_in_hospital,
    COUNT(*) AS patient_count
FROM hospital_data_explore
GROUP BY readmitted
ORDER BY patient_count DESC
""").show(truncate=False)

+----------+--------------------+-------------+
|readmitted|avg_time_in_hospital|patient_count|
+----------+--------------------+-------------+
|no        |4.33                |6810         |
|yes       |4.63                |5655         |
+----------+--------------------+-------------+



This tells us whether patients who are readmitted tend to stay longer in the hospital.

I explored the relationship between time_in_hospital and readmitted by grouping the data by readmission status and comparing and average hospital stay. This is useful because longer stays may indicate more severe complex cases, which could be associated with a higher chance of readmission.

What else could you look for that might be interesting/useful in your dataset? (ex.
seasonality, correlation between columns, etc)

■ Why is it useful to look at for your dataset?

■ Pick one and do the exploration.

■ Explain what you are seeing in the results and how that information is
helpful.

Why this is useful?
Since the dataset is about hospital readmissions so it is very useful to check whther patients with more prior:

1. inpatient visits
2. Emergency visits
3. outpatient visits are more likely to be readmitted.

This matters because:
1. It helps identify patterns linkedin to readmission risk.

2. It gives useful insight before ML

3. It helps explain which features may be important predictors

picked exploration
Explore: pror utilization vs readmitted

In [ ]:
#Compare average utilization by readmission status
spark.sql("""
SELECT
    readmitted,
    ROUND(AVG(n_outpatient), 2) AS avg_n_outpatient,
    ROUND(AVG(n_inpatient), 2) AS avg_inpatient,
    ROUND(AVG(n_emergency), 2) AS avg_emergency,
    COUNT(*) AS patient_count
FROM hospital_data_explore
GROUP BY readmitted
ORDER BY patient_count DESC
""").show(truncate=False)

+----------+----------------+-------------+-------------+-------------+
|readmitted|avg_n_outpatient|avg_inpatient|avg_emergency|patient_count|
+----------+----------------+-------------+-------------+-------------+
|no        |0.0             |0.34         |0.0          |6810         |
|yes       |0.0             |0.7          |0.0          |5655         |
+----------+----------------+-------------+-------------+-------------+



In [ ]:
#Comparing average hospital stay and medications by readmission status

In [ ]:
spark.sql("""
SELECT
    readmitted,
    ROUND(AVG(time_in_hospital), 2) AS avg_time_in_hospital,
    ROUND(AVG(n_medications), 2) AS avg_n_medications,
    COUNT(*) AS patient_count
FROM hospital_data_explore
GROUP BY readmitted
ORDER BY patient_count DESC
""").show(truncate=False)

+----------+--------------------+-----------------+-------------+
|readmitted|avg_time_in_hospital|avg_n_medications|patient_count|
+----------+--------------------+-----------------+-------------+
|no        |4.33                |15.53            |6810         |
|yes       |4.63                |16.21            |5655         |
+----------+--------------------+-----------------+-------------+



In [ ]:
#Looking at the utilization distribution by readmission class
spark.sql("""
SELECT
    readmitted,
    n_inpatient,
    COUNT(*) AS count
FROM hospital_data_explore
GROUP BY readmitted, n_inpatient
ORDER BY readmitted, n_inpatient
""").show(50,truncate=False)

+----------+-----------+-----+
|readmitted|n_inpatient|count|
+----------+-----------+-----+
|no        |0.0        |5081 |
|no        |1.0        |1212 |
|no        |2.0        |338  |
|no        |2.5        |179  |
|yes       |0.0        |3163 |
|yes       |1.0        |1307 |
|yes       |2.0        |610  |
|yes       |2.5        |575  |
+----------+-----------+-----+



Another useful thing to explore in this dataset is the relationship betwee prior healthcare utilization and readmission status. I chose to compare the average number of outpatient visits, inpatient visits and emergency visits across the readmission categories. This is useful because patients with greater prior healthcare usage may have more severe or chronic conditions, which can increase the chance of readmission. The results help identify whether utlization related variables are important predictrs and provide real world isnight into patient risk patterns.

6. Machine Learning
Using SparkML
Using Spark ML (MLLib with DataFrames → the pyspark.ml python package), use
a Classification or Regression Supervised ML algorithm with your dataset.

■ What ML task and algorithm have you chosen and why? Explain what the algorithm will do with your data.

■ Split your data into testing and training datasets (80% train, 20% test)

■ Fit your chosen model to the training data.

■ Make predictions on the previously unseen test data.

■ How did you do? Show how, both in code (be sure to print) and in words, you measure success and what your chosen metric(s) mean.

ML Task
Classification
We want to predict whether a patient will be readmitted or not.

Algorithm Chosen
Random Forest Classifier
why this is a good choice
Random Forest is a strong choice here because:
1. The dataset has a mix of numeric and categorical features.

2. It can capture non linear relationships.

3. It is sually more robust than a single decision tree

4. It works well for structured healthcare tabular data.

5. It does not require heavy assumptions about feature distributions.

What the algorithm will do
1. age
2. hospital stay length
3. number of procedures
4. number of medications
5. prior outpatient/inpatient/emergency visits.
6. diagnosis categories
7. treatment related categorical columns.
and then predict whether a patient is likely to be readmitted.

In [ ]:
#Creating a final dataframe for ML
df_ml = df_capped

In [ ]:
#Creating a binary target column
from pyspark.sql.functions import when, col, lower, trim

df_ml = df_capped

df_ml = df_ml.withColumn(
    "readmitted_clean",
    lower(trim(col("readmitted")))
)

df_ml = df_ml.withColumn(
    "readmitted_binary",
    when(col("readmitted_clean") == "no", 0).otherwise(1)
)

df_ml.groupBy("readmitted", "readmitted_clean", "readmitted_binary").count().show()


+----------+----------------+-----------------+-----+
|readmitted|readmitted_clean|readmitted_binary|count|
+----------+----------------+-----------------+-----+
|       yes|             yes|                1| 5655|
|        no|              no|                0| 6810|
+----------+----------------+-----------------+-----+



Selecting the numeric comlumns


In [ ]:
numeric_cols = [
    "age_mid",
    "time_in_hospital",
    "n_lab_procedures",
    "n_procedures",
    "n_medications",
    "n_outpatient",
    "n_inpatient",
    "n_emergency"
]
categorical_cols = [
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
    "glucose_test",
    "A1Ctest",
    "change",
    "diabetes_med"
]

print("Numeric columns:", numeric_cols)
print("Categorical columns :", categorical_cols)

Numeric columns: ['age_mid', 'time_in_hospital', 'n_lab_procedures', 'n_procedures', 'n_medications', 'n_outpatient', 'n_inpatient', 'n_emergency']
Categorical columns : ['medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'glucose_test', 'A1Ctest', 'change', 'diabetes_med']


In [ ]:
#Indexing categorical columns
from pyspark.ml.feature import StringIndexer
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

In [ ]:
#One hot encoding the indexed categorical columns
from pyspark.ml.feature import OneHotEncoder
indexed_cols = [c + "_index" for c in categorical_cols]
encoded_cols = [c + "_encoded" for c in categorical_cols]

encoder = OneHotEncoder(
    inputCols = indexed_cols,
    outputCols=encoded_cols,

)

In [ ]:
#Assembling all features into one vector
from pyspark.ml.feature import VectorAssembler
feature_cols = numeric_cols + encoded_cols

assembler = VectorAssembler(
    inputCols = feature_cols,
    outputCol="features"
)


In [ ]:
#Defining the classifier
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(
    labelCol ="readmitted_binary",
    featuresCol="features",
    numTrees=100,
    maxDepth=8,
    seed = 42
)

In [ ]:
#Building the pipeline
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=indexers + [encoder, assembler, rf])

In [ ]:
#Splitting data into training and test sets
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed = 42)
print("Training rows :", train_df.count())
print("Testing rows : ", test_df.count())

Training rows : 10025
Testing rows :  2440


In [ ]:
#Fitting the model
model = pipeline.fit(train_df)
print("Model training completed.")

Model training completed.


In [ ]:
#Making predictions on unseen test data
predictions = model.transform(test_df)

In [ ]:
predictions.select(
    "readmitted",
    "readmitted_binary",
    "prediction",
    "probability"
).show(20, truncate=False)

+----------+-----------------+----------+----------------------------------------+
|readmitted|readmitted_binary|prediction|probability                             |
+----------+-----------------+----------+----------------------------------------+
|no        |0                |0.0       |[0.6568254966912919,0.3431745033087082] |
|no        |0                |0.0       |[0.581286772361252,0.418713227638748]   |
|no        |0                |0.0       |[0.6243392548161193,0.37566074518388065]|
|no        |0                |0.0       |[0.6011336328435183,0.3988663671564817] |
|yes       |1                |0.0       |[0.6310005742348802,0.36899942576511974]|
|no        |0                |0.0       |[0.7029677014809902,0.2970322985190098] |
|yes       |1                |0.0       |[0.6039779558108253,0.3960220441891747] |
|no        |0                |0.0       |[0.7085640032728505,0.29143599672714954]|
|yes       |1                |0.0       |[0.5218853210033093,0.4781146789966907] |
|no 

In [ ]:
#Evaluate the model
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol = "readmitted_binary",
    predictionCol="prediction",
    metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol = "readmitted_binary",
    predictionCol="prediction",
    metricName="f1"
)
precision_eval = MulticlassClassificationEvaluator(
    labelCol = "readmitted_binary",
    predictionCol="prediction",
    metricName="weightedPrecision"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="readmitted_binary",
    predictionCol="prediction",
    metricName="weightedRecall"
)

auc_eval = BinaryClassificationEvaluator(
    labelCol="readmitted_binary",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"

)
accuracy = accuracy_eval.evaluate(predictions)
f1 = f1_eval.evaluate(predictions)
precision = precision_eval.evaluate(predictions)
recall = recall_eval.evaluate(predictions)
auc = auc_eval.evaluate(predictions)

print("Accuracy : ", accuracy)
print("F1 score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("AUC:", auc)

Accuracy :  0.6057377049180328
F1 score: 0.5829090799183346
Precision: 0.6159672142994562
Recall: 0.6057377049180328
AUC: 0.636088562179424


In [ ]:
#Confusion matrix style summary
predictions.groupBy("readmitted_binary", "prediction").count().orderBy("readmitted_binary", "prediction").show()

+-----------------+----------+-----+
|readmitted_binary|prediction|count|
+-----------------+----------+-----+
|                0|       0.0| 1061|
|                0|       1.0|  232|
|                1|       0.0|  730|
|                1|       1.0|  417|
+-----------------+----------+-----+



How do you measure success?
I measured the model performance using:

1. Accuracy: The proportion of correct predictions overall here about 60.8% of all the predictions were correct.

2. Precision: Out of all patients predicted as readmitted how many were actually readmitted. Here about 61.9% were actually readmitted.

3. Recall: The model correctly identified about 60.8% of the actual readmitted cases.

4. F1 Score : This summarizes the balance between precision and recall showing the model the model has moderate classification performance.

5. AUC  0.6341 indicates model has modest ability to distinguish between readmitted and non readmitted patients across different thresholds.

Overall these results show that the Random Forest moel performs moderatley well but not perectly. It captures some meaningful patterns in the data though there is still room for improvement.

Take the graph datastore you created in the Graph Database Homework and import that graph as a Spark Graph DataFrame.

■ Use a Spark Connector to connect to your Neo4j data source.

● You can either run Neo4j locally and import from that, or put your sample data in AuraDB (Neo4j cloud) and pull from there.

In [ ]:
#Installing SparkGraphFrames
!pip install graphframes

In [ ]:
#Installing the neo4jgraph database
! curl -fsSL https://debian.neo4j.com/neotechnology.gpg.key | sudo gpg --dearmor -o /usr/share/keyrings/neo4j.gpg
! echo "deb [signed-by=/usr/share/keyrings/neo4j.gpg] https://debian.neo4j.com stable latest" | sudo tee -a /etc/apt/sources.list.d/neo4j.list

gpg: cannot open '/dev/tty': No such device or address
curl: (23) Failed writing body
deb [signed-by=/usr/share/keyrings/neo4j.gpg] https://debian.neo4j.com stable latest


In [ ]:
! sudo apt-get update

Hit:1 https://debian.neo4j.com stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Target Packages (latest/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/neo4j.list:1 and /etc/apt/sources.list.d/neo4j.list:2
W: Target Packages (latest/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/neo4j.list:1 and /etc/apt/sources.list.d/neo4j.list:2
W: Target Pac

In [ ]:
! sudo apt-get install neo4j

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
neo4j is already the newest version (1:2026.03.1).
0 upgraded, 0 newly installed, 0 to remove and 44 not upgraded.
W: Target Packages (latest/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/neo4j.list:1 and /etc/apt/sources.list.d/neo4j.list:2
W: Target Packages (latest/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/neo4j.list:1 and /etc/apt/sources.list.d/neo4j.list:2
W: Target Packages (latest/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/neo4j.list:1 and /etc/apt/sources.list.d/neo4j.list:3
W: Target Packages (latest/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/neo4j.list:1 and /etc/apt/sources.list.d/neo4j.list:3
W: Target Packages (latest/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/neo4j.list:1 and /etc/apt/sources.list.d/neo4j

In [ ]:
! neo4j-admin dbms set-initial-password Rhutam@123

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Changed password for user 'neo4j'. IMPORTANT: this change will only take effect if performed before the database is started for the first time.


In [ ]:
! service neo4j restart

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Stopping Neo4j............ stopped.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Directories in use:
home:         /var/lib/neo4j
config:       /etc/neo4j
logs:         /var/log/neo4j
plugins:      /var/lib/neo4j/plugins
import:       /var/lib/neo4j/import
data:         /var/lib/neo4j/data
certificates: /var/lib/neo4j/certificates
licenses:     /var/lib/neo4j/licenses
run:          /v

In [ ]:
! ps -ax | grep neo4j

  42821 ?        Sl     0:22 /usr/lib/jvm/java-21-openjdk-amd64/bin/java -cp /var/lib/neo4j/plugins/*:/etc/neo4j/*:/usr/share/neo4j/lib/* -XX:+UseG1GC -XX:-OmitStackTraceInFastThrow -XX:+AlwaysPreTouch -XX:+UnlockExperimentalVMOptions -XX:+TrustFinalNonStaticFields -XX:+DisableExplicitGC -Djdk.nio.maxCachedBufferSize=1024 -Dio.netty.tryReflectionSetAccessible=true -Dio.netty.leakDetection.level=DISABLED -Djdk.tls.ephemeralDHKeySize=2048 -Djdk.tls.rejectClientInitiatedRenegotiation=true -XX:FlightRecorderOptions=stackdepth=256 -XX:+UnlockDiagnosticVMOptions -XX:+DebugNonSafepoints --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --enable-native-access=ALL-UNNAMED -Dorg.neo4j.shaded.lucene9.vectorization.upperJavaFeatureVersion=25 -Dlog4j2.disable.jmx=true -Dlog4j.layout.jsonTemplate.maxStringLength=32768 -Dfile.encoding=UTF-8 org.neo4j.server.Neo4jCommu

In [ ]:
from pyspark.sql import SparkSession

# The previous connector was for Spark 3.x. Updating to Spark 4.x compatible version (5.5.0_for_spark_4).
# The GraphFrames dependency (0.8.4-spark3.5-s_2.12) is for Spark 3.5 and Scala 2.12.
# If it causes issues later, it might need further adjustment, as Spark 4 typically uses Scala 2.13.
packages = [
    "org.neo4j:neo4j-connector-apache-spark_2.12:5.3.0_for_spark_3", # Changed to Spark 3.x, Scala 2.12 compatible connector
    "graphframes:graphframes:0.8.4-spark3.5-s_2.12"
]
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("DATA605 Spark GraphFrames Example")
    .config("spark.jars.packages", ",".join(packages))
    .config("neo4j.url", "neo4j://localhost:7687")
    .config("neo4j.authentication.basic.username", "neo4j")
    .config("neo4j.authentication.basic.password", "Rhutam@123")
    .config("neo4j.database", "neo4j")
    .getOrCreate()
)

In [ ]:
print(spark.sparkContext.appName)

DATA605 Spark GraphFrames Example


In [ ]:
#Quick check before importing with spark
!pip install neo4j
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "Rhutam@123"

driver = GraphDatabase.driver(uri, auth=(username, password))
driver.verify_connectivity()
print("Connected to Neo4j successfully.")

with driver.session() as session:
    result = session.run("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(*) AS count
    ORDER BY label
    """)
    for row in result:
        print(row)

driver.close()

Connected to Neo4j successfully.


In [ ]:
#Using spark to connect to neo4j
uri = "bolt://localhost:7687"
username = "neo4j"
password = "Rhutam@123"

test_query = """
CALL {
  MATCH (n)
  RETURN n
  LIMIT 5
}
RETURN n
"""

df = spark.read \
    .format("org.neo4j.spark.DataSource") \
    .option("url", uri) \
    .option("authentication.type", "basic")\
    .option("authentication.basic.username", username) \
    .option("authentication.basic.password", password) \
    .option("query", test_query) \
    .load()

df.show(truncate=False)


Reading the graph into Spark DataFrames

In [ ]:
#vertices DataFrame

In [ ]:
vertices_query = """
MATCH(n)
RETURN
  toString(id(n)) AS id,
  labels(n)[0] AS label,
  coalesce(
      toString(n.user_id),
      n.region_name,
      toString(n.plant_id),
      n.disease_name,
      toString(n.diagnosis_id),
      toString(n.treatment_id)
  )AS entity_key
  """
vertices_df = spark.read \
      .format("org.neo4j.spark.DataSource") \
      .option("url", uri) \
      .option("authentication.basic.username", username) \
      .option("authentication.basic.password", password) \
      .option("query", vertices_query) \
      .load()

vertices_df.show(truncate=False)
print("Vertex count:", vertices_df.count())

In [ ]:
#Edges DataFrame
edges_query = """
MATCH (s)-[r]->(t)
RETURN
  to_string(id(s)) AS src,
  toString(id(t)) AS dst,
  type(r) AS relationship
"""

edges_df = spark.read \
      .format("org.neo4j.spark.DataSource") \
      .option("url", uri) \
      .option("authentication.basic.username", username) \
      .option("authentication.basic.password", password) \
      .option("authentication.basic.password", password) \
      .option("query", edges_query) \
      .load()
edges_df.show(truncate=False)
print("Edge count:", edges_df.count())

I Used the Neo4j Spark Connector to connect Spark  to my local Neo4j datastore that I created in the Graph Database homework. I imported the graph into Spark as two DataFrames a vertices DataFrame and edges DataFrame. I then used these DataFrames to create a GraphFrame in spark. The vertices represented the entities in my PlantVitals graph such as users, regions, plants, diseases, diagnosis and treatments while the edges represented relationships such as lives_in, owns, grown_in, has_diagnosis,identified_as and recommends.